# CMI Problematic Internet Use — Handcrafted Autocorrelation Baseline & `SequenceExtractor`

This notebook implements a **100% handcrafted, zero-ML baseline** for the **Child Mind Institute — Problematic Internet Use** competition (metric: **Quadratic Weighted Kappa** on ordinal `sii` in `{0, 1, 2, 3}`), structured in the style of `multi-branch_version_3.ipynb`:
- **Data Augmentation (`tsaug`)**: Time-warping, cropping, multiplicative drift, jitter, additive noise, time-shifting, temporal masking, and channel dropout via `TSAugmentor`.
- **Feature Extraction (`SequenceExtractor`)**: Modular multi-library extraction across `tsfel`, `tsfresh`, `tsflex`, `pycatch22` / `sktime`, `seglearn`, `statsmodels`, `scipy.signal`, `PyWavelets`, and handcrafted autocorrelation, circadian (`OpenStrap/analytics`, `cr7_kimmitch`), and sleep-architecture (`mad-lab-fau/sleep_analysis`) descriptors with leak-free missing-data imputation.
- **Handcrafted Estimator (`HandcraftedSequenceBaselineEstimator`)**: Deterministic domain scoring (`autocorr_spectral_rule`, `weighted_z_index`, `multiplicative_hazard`) with ordinal QWK threshold calibration.
- **Separate Search Spaces**: Edit `GRID_PARAM_SPACE` or `BAYESIAN_PARAM_SPACE` in their own dedicated cells under `# Params` with zero `.update()` calls.


In [ ]:
import os
import sys
import json
import warnings
import importlib
from pathlib import Path
from datetime import datetime

_py_env = Path(sys.executable).parent
_lib_bin = _py_env / 'Library' / 'bin'
if _lib_bin.is_dir() and str(_lib_bin) not in os.environ.get('PATH', ''):
    os.environ['PATH'] = str(_lib_bin) + os.pathsep + str(_py_env) + os.pathsep + os.environ.get('PATH', '')

import numpy as np
import pandas as pd
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

warnings.filterwarnings('ignore')

# Local path routing
current_dir = os.getcwd()
workspace_root = current_dir
if os.path.basename(current_dir).lower() == 'notebooks':
    workspace_root = os.path.dirname(current_dir)

for _folder in ('Src', 'src'):
    _src_path = os.path.join(workspace_root, _folder)
    if os.path.isdir(_src_path) and _src_path not in sys.path:
        sys.path.insert(0, _src_path)
if workspace_root not in sys.path:
    sys.path.insert(0, workspace_root)

# Robust Kaggle path routing (handles nested folders and symlinks from zip uploads)
if os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input', followlinks=True):
        dirs[:] = [d for d in dirs if not d.endswith('.parquet') and not d.startswith('id=')]
        if 'baseline_utils.py' in files or 'base_utils.py' in files:
            if root not in sys.path:
                sys.path.insert(0, root)

from sklearn.model_selection import GridSearchCV, StratifiedKFold, StratifiedShuffleSplit
from sklearn.pipeline import Pipeline

try:
    import skopt
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
except ImportError:
    Categorical = Integer = Real = None
    SKOPT_AVAILABLE = False

import baseline_utils
import base_utils
importlib.reload(baseline_utils)
importlib.reload(base_utils)

from base_utils import (
    DEFAULT_SEQUENCE_EXTRACTOR_PARAMS,
    HandcraftedSequenceBaselineEstimator,
    HierarchicalBayesSearchCV,
    NoopActigraphyOptions,
    SequenceExtractor,
    TSAugmentor,
    build_quick_results_df,
    candidate_accuracy_pct,
    candidate_score,
    deduplicate_param_grid,
    evaluate_holdout,
    export_submission,
    find_data_root,
    library_status,
    load_cmi_dataset,
    prepare_bayesian_space,
    refit_valid_candidate,
    run_validation_searches,
)

print('Imports loaded successfully')
print('Library availability:', library_status())


# Setup

In [ ]:
experiment_name = 'cmi_piu_handcrafted_baseline'
TARGET_COL = 'sii'

# Set True for a quick sampled smoke test (37 subjects); set False for full train.csv
use_sample_data = False
sample_max_subjects = 37
save_results = not use_sample_data
keep_history = False

search_mode = 'bayes'  # Choose 'bayes' or 'grid'
random_state = 42
n_splits = 5           # 1 -> StratifiedShuffleSplit; >=2 -> StratifiedKFold

train_size = 0.8
n_iter = 8
verbose = 1
error_score = -1.0
n_jobs = 1
refit = refit_valid_candidate

search_mode = str(search_mode).strip().lower()
if search_mode not in ('bayes', 'grid'):
    raise ValueError("search_mode must be 'bayes' or 'grid'")

results_dir = (
    Path(workspace_root) / f'results_{experiment_name}_{TARGET_COL}_{search_mode}'
    if save_results
    else None
)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

if n_splits <= 1:
    cv_object = StratifiedShuffleSplit(
        n_splits=1,
        test_size=(1.0 - train_size),
        random_state=random_state,
    )
else:
    cv_object = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

VALIDATION_PARAM_SPACE = {
    'validation_mode': ['stratified_subject'],
}


# Prep

In [ ]:
data_root = find_data_root()
train_df = load_cmi_dataset(
    data_root=data_root,
    split='train',
    dropna_target=True,
    max_subjects=(sample_max_subjects if use_sample_data else None),
    random_state=random_state,
)
print(f'Using {data_root}: {len(train_df)} labeled subjects (actigraphy={int(train_df["has_actigraphy"].sum())})')


In [ ]:
experiment_df = train_df
del train_df
print('Available subjects:', len(experiment_df))


In [ ]:
extractor = SequenceExtractor(
    noop=NoopActigraphyOptions(),
    **DEFAULT_SEQUENCE_EXTRACTOR_PARAMS,
)

clf = HandcraftedSequenceBaselineEstimator(
    extractor=extractor,
    augmentor=TSAugmentor(prob=0.0, per_aug_prob=0.5, random_state=random_state),
    scoring_rule='autocorr_spectral_rule',
    primary_target=TARGET_COL,
    threshold_calibration='optimize_qwk',
    verbose=verbose,
    random_state=random_state,
)
pipeline = Pipeline([('estimator', clf)])


# Params

Search spaces are organised into separate, self-contained cells (`GRID_PARAM_SPACE` and `BAYESIAN_PARAM_SPACE`) with zero `.update()` calls.


## Grid

In [ ]:
GRID_PARAM_SPACE = {
    # ----------------------------------------------------------
    # AUGMENTOR
    # ----------------------------------------------------------
    'estimator__augmentor__prob': [0.0],
    'estimator__augmentor__random_state': [random_state],

    # ----------------------------------------------------------
    # SEQUENCE EXTRACTOR & IMPUTATION
    # ----------------------------------------------------------
    'estimator__extractor__bin_minutes': [15],
    'estimator__extractor__resample_method': ['anti_alias_mean'],
    'estimator__extractor__static_backend': ['tsfel+tsfresh', 'none'],
    'estimator__extractor__tsfel_domains': ['statistical|temporal|spectral'],
    'estimator__extractor__tsfresh_profile': ['minimal'],
    'estimator__extractor__seglearn_feature_set': ['safe_all'],
    'estimator__extractor__catch22_catch24': [True],
    'estimator__extractor__clip_enmo': [4.0],
    'estimator__extractor__non_wear_policy': ['none', 'mask', 'impute_diurnal'],
    'estimator__extractor__interp_mode': ['none', 'bounded_linear', 'diurnal_median'],
    'estimator__extractor__max_gap_bins': [4],
    'estimator__extractor__missing_feature_strategy': ['none', 'group_age_median', 'knn'],
    'estimator__extractor__impute_knn_neighbors': [5],
    'estimator__extractor__add_missing_indicators': [True],
    'estimator__extractor__linear_acc_mode': ['rolling_mean'],
    'estimator__extractor__motion_filter_mode': ['none', 'kalman'],
    'estimator__extractor__acf_detrend': ['linear'],
    'estimator__extractor__acf_bias': ['unbiased'],
    'estimator__extractor__sedentary_threshold_g': [0.025],
    'estimator__extractor__feature_families': [
        'none',
        'autocorr|circadian|sleep_arch|motion|spectral|stft|wavelet|seglearn|catch22|tsfel|tsfresh|tsflex|prsa_cusum|branches|compliance',
    ],

    # ----------------------------------------------------------
    # HANDCRAFTED SCORING RULE & CALIBRATION
    # ----------------------------------------------------------
    'estimator__scoring_rule': ['weighted_z_index', 'autocorr_spectral_rule', 'multiplicative_hazard'],
    'estimator__threshold_calibration': ['optimize_qwk'],
    'estimator__random_state': [random_state],
}


## Bayesian

In [ ]:
# BAYESIAN SEARCH ONLY
BAYESIAN_PARAM_SPACE = dict()
if SKOPT_AVAILABLE:
    tsfel_minimal = {
        'statistical': {
            'Mean': {'use': 'yes'},
            'Variance': {'use': 'yes'},
            'Max': {'use': 'yes'},
            'Min': {'use': 'yes'},
        }
    }
    tsfel_kinematic = {
        'statistical': {
            'Mean': {'use': 'yes'},
            'Variance': {'use': 'yes'},
            'Root mean square': {'use': 'yes'},
        },
        'temporal': {
            'Peak to peak distance': {'use': 'yes'},
            'Zero crossing rate': {'use': 'yes'},
            'Slope': {'use': 'yes'},
        },
    }

    BAYESIAN_PARAM_SPACE = {
        # ----------------------------------------------------------
        # AUGMENTOR (tsaug)
        # ----------------------------------------------------------
        'estimator__augmentor__prob': Categorical([0.0, 0.25, 0.5]),
        'estimator__augmentor__per_aug_prob': Categorical([0.0, 0.5]),
        'estimator__augmentor__n_copies': Categorical([0, 1]),
        'estimator__augmentor__jitter_sigma': Categorical([0.0, 0.02]),
        'estimator__augmentor__noise_std': Categorical([0.0, 0.02]),
        'estimator__augmentor__scaling_sigma': Categorical([0.0, 0.05]),
        'estimator__augmentor__warp_sigma': Categorical([0.0, 0.05]),
        'estimator__augmentor__time_warp_max_speed_ratio': Categorical([1.0, 1.25]),
        'estimator__augmentor__time_shift_frac': Categorical([0.0, 0.04]),
        'estimator__augmentor__temporal_num_masks': Categorical([0, 1]),
        'estimator__augmentor__temporal_mask_frac': Categorical([0.0, 0.05]),
        'estimator__augmentor__channel_drop_prob': Categorical([0.0, 0.1]),
        'estimator__augmentor__random_state': Categorical([random_state]),

        # ----------------------------------------------------------
        # EXTRACTOR — Cleaning, Resampling & Imputation
        # ----------------------------------------------------------
        'estimator__extractor__bin_minutes': Categorical([15, 30]),
        'estimator__extractor__resample_method': Categorical(['none', 'anti_alias_mean']),
        'estimator__extractor__non_wear_policy': Categorical(['none', 'mask', 'impute_diurnal']),
        'estimator__extractor__interp_mode': Categorical(['none', 'bounded_linear', 'bounded_pchip', 'diurnal_median']),
        'estimator__extractor__min_bin_wear_frac': Categorical([0.0, 0.25]),
        'estimator__extractor__max_gap_bins': Categorical([0, 2, 4, 8]),
        'estimator__extractor__impute_long_gaps': Categorical([False, True]),
        'estimator__extractor__min_valid_wear_days': Categorical([0.0, 1.0]),
        'estimator__extractor__clip_enmo': Categorical([0.0, 4.0]),
        'estimator__extractor__linear_acc_mode': Categorical(['none', 'rolling_mean', 'highpass_sos']),
        'estimator__extractor__baseline_window_bins': Categorical([0, 16]),
        'estimator__extractor__motion_filter_mode': Categorical(['none', 'kalman']),
        'estimator__extractor__use_dead_reckoning': Categorical([False, True]),
        'estimator__extractor__smooth_window_bins': Categorical([0, 4]),
        'estimator__extractor__smooth_alpha': Categorical([0.0, 0.3]),

        # ----------------------------------------------------------
        # EXTRACTOR — Autocorrelation, Spectral, Wavelet & Open-Source Libraries
        # ----------------------------------------------------------
        'estimator__extractor__acf_detrend': Categorical(['none', 'linear', 'constant']),
        'estimator__extractor__acf_bias': Categorical(['unbiased', 'biased']),
        'estimator__extractor__sedentary_threshold_g': Categorical([0.025, 0.020, 0.035]),
        'estimator__extractor__fundamental_ratio': Categorical([0.85, 0.80]),
        'estimator__extractor__min_acf_strength': Categorical([0.0, 0.08]),
        'estimator__extractor__stft_window_days': Categorical([2.0, 3.0]),
        'estimator__extractor__stft_overlap_ratio': Categorical([0.0, 0.5]),
        'estimator__extractor__stft_window_type': Categorical(['none', 'hann']),
        'estimator__extractor__cwt_wavelet': Categorical(['none', 'morl', 'mexh']),
        'estimator__extractor__cwt_n_scales': Categorical([0, 16]),
        'estimator__extractor__dwt_wavelet': Categorical(['none', 'db4']),
        'estimator__extractor__dwt_level': Categorical([0, 4]),
        'estimator__extractor__seglearn_window_days': Categorical([1.0, 2.0, 3.0]),
        'estimator__extractor__seglearn_overlap': Categorical([0.0, 0.5]),
        'estimator__extractor__seglearn_feature_set': Categorical(['none', 'safe_all', 'base', 'emg_kinematic']),
        'estimator__extractor__catch22_catch24': Categorical([False, True]),
        'estimator__extractor__static_backend': Categorical(['none', 'tsfel+tsfresh', 'tsfel', 'tsfresh']),
        'estimator__extractor__tsfel_domains': Categorical(['none', 'statistical|temporal|spectral']),
        'estimator__extractor__tsfel_features': Categorical([
            'None',
            'none',
            json.dumps(tsfel_minimal, sort_keys=True),
            json.dumps(tsfel_kinematic, sort_keys=True),
        ]),
        'estimator__extractor__tsfresh_profile': Categorical(['none', 'minimal', 'efficient']),
        'estimator__extractor__tsflex_windows': Categorical([
            'none',
            json.dumps(['2h', '6h']),
            json.dumps(['2h', '6h', '12h']),
        ]),
        'estimator__extractor__tsflex_strides': Categorical(['2h', '4h']),
        'estimator__extractor__include_tabular_engineered': Categorical([False, True]),
        'estimator__extractor__harmonize_instruments': Categorical([False, True]),
        'estimator__extractor__missing_feature_strategy': Categorical(['none', 'keep_nan', 'group_age_median', 'median', 'knn', 'iterative']),
        'estimator__extractor__add_missing_indicators': Categorical([False, True]),
        'estimator__extractor__impute_knn_neighbors': Integer(3, 9),
        'estimator__extractor__feature_families': Categorical([
            'none',
            'autocorr|circadian|sleep_arch|compliance',
            'autocorr|circadian|sleep_arch|motion|spectral|stft|wavelet|seglearn|catch22|tsfel|tsfresh|tsflex|prsa_cusum|branches|compliance',
        ]),

        # ----------------------------------------------------------
        # HANDCRAFTED DOMAIN RULES & PRIORS
        # ----------------------------------------------------------
        'estimator__scoring_rule': Categorical(['autocorr_spectral_rule', 'weighted_z_index', 'multiplicative_hazard']),
        'estimator__w_internet_hours': Real(0.0, 2.2),
        'estimator__w_age': Real(0.0, 1.2),
        'estimator__w_age_internet': Real(0.0, 1.6),
        'estimator__w_sleep_disturbance': Real(0.0, 1.2),
        'estimator__w_inactivity': Real(0.0, 0.8),
        'estimator__w_cgas_deficit': Real(0.0, 0.8),
        'estimator__w_acf_irregularity': Real(0.0, 1.0),
        'estimator__w_sedentary_acf': Real(0.0, 1.0),
        'estimator__w_sri_irregularity': Real(0.0, 1.0),
        'estimator__w_nocturnal_ratio': Real(0.0, 0.9),
        'estimator__w_upright_night': Real(0.0, 0.9),
        'estimator__w_acrophase_delay': Real(0.0, 0.8),
        'estimator__w_period_drift': Real(0.0, 0.7),
        'estimator__w_stft_flux': Real(0.0, 0.7),
        'estimator__w_cwt_ultradian': Real(0.0, 0.7),
        'estimator__w_seglearn_kinematic': Real(0.0, 0.7),
        'estimator__w_cusum_prsa': Real(0.0, 0.6),
        'estimator__w_sleep_fragmentation': Real(0.0, 0.7),
        'estimator__coupling_alpha': Real(0.0, 0.8),
        'estimator__threshold_calibration': Categorical(['optimize_qwk', 'quantile', 'none']),
        'estimator__random_state': Categorical([random_state]),
    }
    BAYESIAN_PARAM_SPACE = prepare_bayesian_space(BAYESIAN_PARAM_SPACE)

for space_name, candidate_space in (
    ('grid', GRID_PARAM_SPACE),
    ('bayesian', BAYESIAN_PARAM_SPACE),
):
    unknown_params = set(candidate_space) - set(pipeline.get_params(deep=True))
    if unknown_params:
        raise ValueError(f'{space_name}: unknown={sorted(unknown_params)}')

if search_mode == 'bayes':
    param_space = BAYESIAN_PARAM_SPACE
else:
    param_space = GRID_PARAM_SPACE
print(f'Validated {len(GRID_PARAM_SPACE)} Grid and {len(BAYESIAN_PARAM_SPACE)} Bayesian parameters.')


# Train

In [ ]:
if search_mode == 'bayes':
    if not SKOPT_AVAILABLE:
        raise ImportError('scikit-optimize is required for Bayesian search')
    search = HierarchicalBayesSearchCV(
        estimator=pipeline,
        search_spaces=param_space,
        n_iter=n_iter,
        scoring=candidate_score,
        cv=cv_object,
        refit=refit,
        random_state=random_state,
        verbose=verbose,
        error_score=error_score,
        n_jobs=n_jobs,
    )
else:
    deduped_grid = deduplicate_param_grid(param_space, base_params=pipeline.get_params(deep=True))
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=deduped_grid,
        scoring={'accuracy_pct': candidate_accuracy_pct, 'qwk': candidate_score},
        cv=cv_object,
        refit=refit,
        verbose=verbose,
        error_score=error_score,
        n_jobs=n_jobs,
    )

validation_runs = run_validation_searches(
    search=search,
    experiment_df=experiment_df,
    target_col=TARGET_COL,
    validation_param_space=VALIDATION_PARAM_SPACE,
    train_size=train_size,
    n_splits=n_splits,
    random_state=random_state,
    results_dir=results_dir,
    experiment_name=experiment_name,
    search_mode=search_mode,
    timestamp=timestamp,
    save_results=save_results,
    keep_history=keep_history,
)

last_run = validation_runs['last_run']
protocol_results = validation_runs['protocol_results']
experiment_succeeded = bool(last_run.get('experiment_succeeded', False))
if experiment_succeeded:
    search = last_run['search']
    X_train = last_run['X_train']
    y_train = last_run['y_train']
    X_test = last_run['X_test']
    y_test = last_run['y_test']
    eval_results = last_run['holdout_results']


# Analysis

In [ ]:
if experiment_succeeded:
    print('Last validation protocol:', last_run['validation_mode'])
    print('Holdout normal score (%):', eval_results.get('accuracy_pct'))
    print('Holdout competition score (QWK):', eval_results['competition_score'])
pd.DataFrame(protocol_results)


In [ ]:
print('Experiments CSV:', validation_runs['experiments_path'])
print('Summary JSON:', validation_runs['summary_path'])


In [ ]:
if experiment_succeeded:
    test_df = load_cmi_dataset(data_root=data_root, split='test', dropna_target=False)
    sample_sub_path = data_root / 'sample_submission.csv'
    sub_workspace_path = (Path(workspace_root) / 'submission.csv') if save_results else None
    submission_df = export_submission(
        search.best_estimator_,
        test_df,
        output_path=sub_workspace_path,
        sample_sub_path=sample_sub_path if sample_sub_path.exists() else None,
    )
    print('Submission written to:', sub_workspace_path)
    display(submission_df.head(10))
